<a href="https://colab.research.google.com/github/ishaanrai-hub/IIT-Hyd-Projects-Machine-learning-/blob/main/Predict_customer_purchase_behavior.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install scikit-learn pandas numpy shap

In [2]:
# 📊 Data Setup (Same Structure)
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

In [3]:
# Simulated Dataset
np.random.seed(42)
N = 100_000

data = pd.DataFrame({
    "age": np.random.randint(18, 70, N),
    "income": np.random.randint(20_000, 150_000, N),
    "time_on_site": np.random.exponential(10, N),
    "country": np.random.choice(["US", "UK", "DE", "FR", "IN"], N),
    "device_type": np.random.choice(["mobile", "desktop", "tablet"], N),
    "membership_tier": np.random.choice(["free", "silver", "gold"], N),
    "purchased": np.random.randint(0, 2, N)
})


In [4]:
print(data)

       age  income  time_on_site country device_type membership_tier  \
0       56   48339      3.474361      UK      tablet            gold   
1       69   80673     24.542901      FR     desktop            gold   
2       46   35320      9.093409      US      mobile            gold   
3       32   54228      9.892634      UK      mobile          silver   
4       60  121058     12.000566      DE     desktop            gold   
...    ...     ...           ...     ...         ...             ...   
99995   30   44825      1.325457      US     desktop          silver   
99996   53   45620     22.734856      DE     desktop            free   
99997   33  117067     11.955177      IN      tablet          silver   
99998   57   38593     21.123341      US      mobile            gold   
99999   20   28233      6.192363      UK      tablet          silver   

       purchased  
0              0  
1              1  
2              0  
3              0  
4              0  
...          ...  
99

In [5]:
# Feature Engineering
X = data.drop("purchased", axis=1)
y = data["purchased"]

categorical_features = ["country", "device_type", "membership_tier"]
numerical_features = ["age", "income", "time_on_site"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numerical_features)
    ]
)

In [6]:
# Logistic Regression Model
model = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    max_iter=1000,
    n_jobs=-1
)

In [7]:
# 🔗 Pipeline
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", model)
])


In [8]:
# Train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['country', 'device_type',
                                                   'membership_tier']),
                                                 ('num', StandardScaler(),
                                                  ['age', 'income',
                                                   'time_on_site'])])),
                ('classifier', LogisticRegression(max_iter=1000, n_jobs=-1))])

In [9]:
# Evaluate
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.5005
ROC-AUC: 0.5001332650652999


In [10]:
# Global Interpretation (Top Drivers)
feature_names = (
    pipeline.named_steps["preprocessing"]
    .get_feature_names_out()
)

coefficients = pipeline.named_steps["classifier"].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
}).sort_values(by="coefficient", key=abs, ascending=False)

print(coef_df.head(10))

                      feature  coefficient
8   cat__membership_tier_free     0.024944
2             cat__country_IN     0.024272
9   cat__membership_tier_gold    -0.019764
4             cat__country_US    -0.019029
7     cat__device_type_tablet    -0.017765
6     cat__device_type_mobile     0.016960
12                num__income     0.010105
0             cat__country_DE     0.008848
1             cat__country_FR    -0.007474
3             cat__country_UK    -0.005560


In [11]:
# Local Explanation (Individual Prediction)
# Log-odds explanation
sample = X_test.iloc[[0]]
log_odds = pipeline.decision_function(sample)[0]
probability = pipeline.predict_proba(sample)[0, 1]

print("Log-odds:", log_odds)
print("Purchase probability:", probability)

Log-odds: -0.029866543734055333
Purchase probability: 0.4925339190434121


In [12]:
# Choose Logistic Regression if:

# Stakeholders want simple, transparent explanations

# Latency must be as low as possible

# Model governance or compliance is critical

# Slightly lower accuracy is acceptable